<a href="https://colab.research.google.com/github/Ane-Graciano/ihc/blob/main/HIC_filmes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,204 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 118194 files and directories currently 

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time
import requests

subprocess.Popen("ollama serve", shell=True)

for i in range(10):
    try:
        requests.get("http://localhost:11434")
        print("Ollama rodando")
        break
    except:
        time.sleep(2)

Ollama rodando


In [4]:
# !ollama pull qwen2.5
# !ollama pull llama3.1:8b
# !ollama pull llama3.2:3b
!for i in {1..5}; do ollama pull qwen2.5:3b && break || sleep 5; done

In [5]:
!uv pip install ollama

Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 881ms
Prepared 1 package in 55ms
Installed 1 package in 6ms
 + ollama==0.6.2


In [6]:
!uv pip install rich

Using Python 3.12.13 environment at: /usr
Checked 1 package in 86ms


# Instalar gdown e baixar os dataset

In [7]:
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/1RMKJiCSCL5Vtk4WonvAcldRRfEaEgPr4

Retrieving folder contents
Processing file 1zRw0xYUtvBFxXc_AjIadebR37A3y-NM8 dataset-metadata.json
Processing file 1dmavY-0wSyn84Uwe50OChAs-8hsrovMd tmdb_movies_2021_2025.csv
Processing file 1kAoluof9BRs4u1H7LR8VjtPWC05RBpQk tmdb_movies_2021_2025.parquet
Processing file 1Q1CBMCfJrZiVnU59WAwbV4Qx_Uo108R9 tmdb-movies-dataset-20212025.zip
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1zRw0xYUtvBFxXc_AjIadebR37A3y-NM8
To: /content/dataset_colab/dataset-metadata.json
100% 1.47k/1.47k [00:00<00:00, 4.33MB/s]
Downloading...
From: https://drive.google.com/uc?id=1dmavY-0wSyn84Uwe50OChAs-8hsrovMd
To: /content/dataset_colab/tmdb_movies_2021_2025.csv
100% 84.4M/84.4M [00:01<00:00, 71.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1kAoluof9BRs4u1H7LR8VjtPWC05RBpQk
To: /content/dataset_colab/tmdb_movies_2021_2025.parquet
100% 53.4M/53.4M [00:01<00:00, 42.9MB/s]
Downloading...
Fr

# Carregar a base (recomendado usar parquet por rapidez para manusear dataset)

In [8]:
import pandas as pd
import json
import os
df = pd.read_parquet("/content/dataset_colab/tmdb_movies_2021_2025.parquet")
# display(df)

In [9]:
# import json
# from rich import print
# from ollama import chat
# model = 'granite4:350m'

In [10]:
import sqlite3
from datetime import datetime

DATABASE_NAME = 'movies.db'

In [11]:
import sqlite3

def get_db_connection():
    """Establishes a connection to the SQLite database and confirms success."""
    try:
        conn = sqlite3.connect(DATABASE_NAME)
        conn.row_factory = sqlite3.Row  # Permite acessar colunas pelo nome
        print(f"Conexão com o banco '{DATABASE_NAME}' estabelecida com sucesso!")
        return conn
    except sqlite3.Error as e:
        print(f"Erro ao conectar com o banco: {e}")
        return None

# Teste
conn = get_db_connection()
if conn:
    conn.close()

Conexão com o banco 'movies.db' estabelecida com sucesso!


In [12]:
def drop_table():
    """Drops the movies table if it exists."""
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute('DROP TABLE IF EXISTS movies')

    conn.commit()
    print("Database table 'MOVIES' dropped (if it existed).")

drop_table()

def create_table():
    """Creates the movie table if it doesn't already exist."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS movies (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            original_title TEXT,
            release_date DATE,
            genres TEXT NOT NULL,
            vote_average DECIMAL,
            vote_count INTEGER,
            popularity DECIMAL,
            original_language TEXT,
            overview TEXT,
            poster_url TEXT
        )
    ''')
    conn.commit()
    print("Database table 'MOVIES' ensured.")

create_table()

Conexão com o banco 'movies.db' estabelecida com sucesso!
Database table 'MOVIES' dropped (if it existed).
Conexão com o banco 'movies.db' estabelecida com sucesso!
Database table 'MOVIES' ensured.


In [13]:
def inserir():
  conn = get_db_connection()
  cursor = conn.cursor()
  for _, row in df.iterrows():

      cursor.execute('''
          INSERT INTO movies (id, title, original_title, release_date, genres, vote_average,
                              vote_count, popularity, original_language, overview, poster_url)
          VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
      ''', (
          row['tmdb_id'],
          row['title'],
          row['original_title'],
          row['release_date'],
          row['genres'],
          row['vote_average'],
          row['vote_count'],
          row['popularity'],
          row['original_language'],
          row['overview'],
          row['poster_url']
      ))
  conn.commit()
  conn.close()
  print("Inserido.")
inserir()

Conexão com o banco 'movies.db' estabelecida com sucesso!
Inserido.


In [14]:
def selectAll():
    """Selects all movies from the database and formats them nicely."""
    conn = get_db_connection()
    cursor = conn.cursor()

    query = "SELECT * FROM movies;"
    cursor.execute(query)
    movies = cursor.fetchall()
    conn.close()

    if not movies:
        return "Nenhum filme encontrado."

    # Formata cada linha com "coluna: valor"
    formatted_movies = []
    for row in movies:
        formatted_row = ', '.join(f"{key}: {row[key]}" for key in row.keys())
        formatted_movies.append(formatted_row)

    return '\n\n'.join(formatted_movies)

# Teste
print(selectAll())

A saída de streaming foi truncada nas últimas 5000 linhas.

id: 1619405, title: Becquerel, original_title: Becquerel, release_date: 2025-10-18, genres: , vote_average: 0, vote_count: 0, popularity: 0.0281, original_language: en, overview: Bequerel is a collaboration between composer McKenzie Stubbert and the inventor/visual artist Daniel Vettorazi. Vettorazi used creative coding and data moshing, a form of video manipulation that involves intentionally corrupting the compression data of video files, to create the glitch effects that aligned with Stubbert's original composition. The music was created from acoustic recordings of a piano and a harp, processed and re-composed. The becquerel is the unit of radioactivity in the International System of Units (SI). One becquerel is defined as an activity of one decay per second., poster_url: https://image.tmdb.org/t/p/w500/3qw1oOEdts8ycNb2XlntMAk5gEA.jpg

id: 1619410, title: いちごタルトのいちごだけ 池田ゆうな Aircontrol, original_title: いちごタルトのいちごだけ 池田ゆうな Air

In [15]:
def search_by_name(title: str) -> str:
    conn = get_db_connection()
    cursor = conn.cursor()

    query = "SELECT * FROM movies WHERE title LIKE ?"
    cursor.execute(query, (f"%{title}%",))  # FIX AQUI

    movies = cursor.fetchall()
    conn.close()

    if not movies:
        return "Nenhum filme encontrado."

    return '\n\n'.join(
        ', '.join(f"{k}: {row[k]}" for k in row.keys())
        for row in movies
    )

In [16]:
import ipywidgets as widgets
from IPython.display import display, HTML
from ollama import chat
import sqlite3
import json
import time

# =========================
# TIMER
# =========================
def agora():
    return time.perf_counter()

# =========================
# DATABASE
# =========================
DB_PATH = "movies.db"

def get_db_connection():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

# =========================
# MODEL
# =========================
model = "qwen2.5:3b"

# =========================
# MEMORY (curta para velocidade)
# =========================
messages = [
    {
        "role": "system",
        "content": (
            "Você é um assistente rápido de filmes.\n"
            "Regras:\n"
            "- Sempre use tools para dados de filmes\n"
            "- Nunca invente filmes\n"
            "- Responda de forma curta e direta\n"
        )
    }
]

# =========================
# TOOLS
# =========================
tools = [
    {
        "type": "function",
        "function": {
            "name": "selectAll",
            "description": "Retorna filmes do banco",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_by_name",
            "description": "Busca filmes por título",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"}
                },
                "required": ["title"]
            }
        }
    }
]

# =========================
# UI
# =========================
chat_history = widgets.Output(layout={
    "border": "1px solid #ddd",
    "height": "500px",
    "overflow_y": "auto",
    "padding": "10px"
})

input_box = widgets.Text(placeholder="Pergunte sobre filmes...")
button = widgets.Button(description="Enviar", button_style="primary")

display(widgets.HBox([input_box, button]), chat_history)

def render(role, text):
    label = "Você" if role == "user" else "Assistente"

    html = f"""
    <div style="margin:8px;">
        <b>{label}:</b><br>
        <pre style="white-space: pre-wrap; font-family: Arial;">{text}</pre>
    </div>
    """

    with chat_history:
        display(HTML(html))

# =========================
# TOOLS (rápidas)
# =========================
def selectAll():
    t = agora()

    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT title, release_date, genres, vote_average
        FROM movies
        LIMIT 10
    """)

    rows = cursor.fetchall()
    conn.close()

    print("[DEBUG] Tempo selectAll:", round(agora() - t, 4), "segundos")

    return "\n".join(
        f"{r[0]} | {r[1]} | {r[2]} | {r[3]}"
        for r in rows
    )


def search_by_name(title: str):
    t = agora()

    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT title, release_date, genres, vote_average
        FROM movies
        WHERE title LIKE ?
        LIMIT 10
    """, (f"%{title}%",))

    rows = cursor.fetchall()
    conn.close()

    print("[DEBUG] Tempo search_by_name:", round(agora() - t, 4), "segundos")

    return "\n".join(
        f"TITULO: {r[0]} | DATA: {r[1]} | GENERO: {r[2]} | NOTA: {r[3]}"
        for r in rows
    )

# =========================
# EXECUTOR
# =========================
def execute_tool(tool_call):
    nome = tool_call.function.name
    args = tool_call.function.arguments

    print("[DEBUG] Tool chamada:", nome)

    if isinstance(args, str):
        args = json.loads(args or "{}")

    func = globals().get(nome)

    if not func:
        print("[DEBUG] Tool não encontrada:", nome)
        return "tool não encontrada"

    return func(**args)

# =========================
# AGENTE (FAST MODE)
# =========================
def run_agent(user_input):
    t_total = agora()

    print("[DEBUG] Usuário:", user_input)

    render("user", user_input)
    messages.append({"role": "user", "content": user_input})

    # limita contexto (grande ganho de velocidade)
    if len(messages) > 6:
        messages[:] = [messages[0]] + messages[-5:]

    # =========================
    # LLM 1
    # =========================
    t1 = agora()

    response = chat(
        model=model,
        messages=messages,
        tools=tools,
        options={
            "temperature": 0,
            "num_predict": 180
        }
    )

    print("[DEBUG] Tempo LLM1:", round(agora() - t1, 4), "segundos")

    msg = response.message
    tool_calls = getattr(msg, "tool_calls", None)

    print("[DEBUG] Tool calls:", tool_calls)

    if not tool_calls:
        render("assistant", msg.content or "")
        messages.append({"role": "assistant", "content": msg.content or ""})
        print("[DEBUG] Tempo total:", round(agora() - t_total, 4), "segundos")
        return

    messages.append({
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": tool_calls
    })

    # =========================
    # TOOLS
    # =========================
    for tool_call in tool_calls:
        resultado = execute_tool(tool_call)

        messages.append({
            "role": "tool",
            "name": tool_call.function.name,
            "content": str(resultado)
        })

    # =========================
    # LLM 2 (resposta final)
    # =========================
    t2 = agora()

    response2 = chat(
        model=model,
        messages=messages[-6:],
        options={
            "temperature": 0,
            "num_predict": 200
        }
    )

    print("[DEBUG] Tempo LLM2:", round(agora() - t2, 4), "segundos")

    final = response2.message.content

    render("assistant", final)
    messages.append({"role": "assistant", "content": final})

    print("[DEBUG] Tempo total:", round(agora() - t_total, 4), "segundos")

# =========================
# EVENTS
# =========================
def on_click(_):
    texto = input_box.value.strip()
    input_box.value = ""

    if texto:
        run_agent(texto)

button.on_click(on_click)
input_box.on_submit(on_click)

print("Chat iniciado")

Output(layout=Layout(border='1px solid #ddd', height='500px', overflow_y='auto', padding='10px'))

Chat iniciado
[DEBUG] Usuário: busque o Lightspeed
[DEBUG] Tempo LLM1: 58.6417 segundos
[DEBUG] Tool calls: [ToolCall(function=Function(name='search_by_name', arguments={'title': 'Lightspeed'}))]
[DEBUG] Tool chamada: search_by_name
[DEBUG] Tempo search_by_name: 0.0823 segundos
[DEBUG] Tempo LLM2: 44.6526 segundos
[DEBUG] Tempo total: 103.3881 segundos


In [17]:
print(search_by_name('lightspeed'))

[DEBUG] Tempo search_by_name: 0.1275 segundos
TITULO: Sweet Lightspeed | DATA: 2024-02-04 | GENERO: Drama|Romance | NOTA: 9
TITULO: 'Lightspeed' | DATA: 2025-12-12 | GENERO: Drama | NOTA: 0


In [18]:
#!pkill ollama

Busca por:
*   nome
*   gênero
*   avaliação
*   ano de lançamento
*   diretor
*   streaming
*   quantidade de filmes por ator
*   últimos lançamentos
*   classificação etária
*   premiação oscar
*   temas gerais (ex: filmes sobre superação)









